# Customer Segmentation & Revenue Strategy
## Online Retail II - UK E-commerce analysis


**Business Context:**
-- A UK based e-commerce company allocated budget across
all customer segments equally. The CMO( Cheif marketing Officer )
needs a data-driven insights that can help to reallocate budget 
to high spending customers and re-engaging with at-risk ones.

**Key Questions:**
1. Who are our most valuable customers and what defines them ?
2. Which customers are at risk of churning ?
3. How should marketing budget be redistributed across segments ?
4. At what point do most customers stop purchasing ?


**Dataset** : Online Retail II (UCI ML Repository)
**Dataset_Source** : https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
**Tools** : Python (Pandas, Numpy, Matlplotlib, Seaborn, Scipy, Tableau, Statistics)

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12,6)
plt.rcParams['font.size'] = 12


In [16]:
#Loading data

df1 = pd.read_excel("D:\90_Days\Project-1\online_retail_II.xlsx",sheet_name = 'Year 2009-2010')

In [19]:
df2 = pd.read_excel("D:\90_Days\Project-1\online_retail_II.xlsx",sheet_name = 'Year 2010-2011')

In [25]:
df = pd.concat([df1,df2],ignore_index=True)
print(f"Total records: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

Total records: 1,067,371
Total columns: 8
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 1. Data Inspection

Before performing analysis step, I understand data columns and values,
examine raw data structure, data types, To find missing values, Anomalies

This step prevents from misassumptions.

In [112]:
print(df.head(10))
print("\n")
print(df.tail(10))
print("\n")
print(df.sample(10))
print("\n")
print(df.columns.tolist())
print("\n")

#Datatype
print(df.dtypes)

print(df.describe())
print("\n")
print(df.info(memory_usage='deep'))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
5  489434     22064           PINK DOUGHNUT TRINKET POT         24   
6  489434     21871                  SAVE THE PLANET MUG        24   
7  489434     21523   FANCY FONT HOME SWEET HOME DOORMAT        10   
8  489435     22350                            CAT BOWL         12   
9  489435     22349       DOG BOWL , CHASING BALL DESIGN        12   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United

In [111]:
# Column - by - Column inspection

for col in df.columns:
    print(f"\n{'='*50}")
    print("Column_Name:",col)
    print(f"Column_Type: {df[col].dtype}")
    print(f"Null value Count: {df[col].isnull().sum()}")
    print(f"Null Value Percentage: {df[col].isnull().mean()*100:.2f}%")
    print(f"Unique Values: {df[col].nunique()}")
    if df[col].dtype in ['object','Str']:
        print(f"Sample Values: {df[col].dropna().unique()[:5]}")
    elif df[col].dtype in ['int64','datetime64[ns]','float64']:
        print(f"Min : {df[col].min()}, Max : {df[col].max()}")


Column_Name: Invoice
Column_Type: object
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 53628
Sample Values: [489434 489435 489436 489437 489438]

Column_Name: StockCode
Column_Type: object
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 5305
Sample Values: [85048 '79323P' '79323W' 22041 21232]

Column_Name: Description
Column_Type: object
Null value Count: 4382
Null Value Percentage: 0.41%
Unique Values: 5698
Sample Values: ['15CM CHRISTMAS GLASS BALL 20 LIGHTS' 'PINK CHERRY LIGHTS'
 ' WHITE CHERRY LIGHTS' 'RECORD FRAME 7" SINGLE SIZE '
 'STRAWBERRY CERAMIC TRINKET BOX']

Column_Name: Quantity
Column_Type: int64
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 1057
Min : -80995, Max : 80995

Column_Name: InvoiceDate
Column_Type: datetime64[ns]
Null value Count: 0
Null Value Percentage: 0.00%
Unique Values: 47635
Min : 2009-12-01 07:45:00, Max : 2011-12-09 12:50:00

Column_Name: Price
Column_Type: float64
Null value Count: 0
Null Value P

Column    | Type   | Null | Issues Found
Invoice   | Object |  0   | As cancelled orders Invoice start with 'C', They are valid but we need to ignore them when calculating totals
StockCode | Object |  0   | Some values contain administratvie codes, They are not 'errors' but should be ignored when performing product analysis



In [152]:
print(df.sample(10))

        Invoice StockCode                         Description  Quantity  \
611391   543530     84946     ANTIQUE SILVER TEA GLASS ETCHED         4   
664642   548308     22501          PICNIC BASKET WICKER LARGE         2   
754588   557014     21977  PACK OF 60 PINK PAISLEY CAKE CASES        24   
623471   544664     21531        RED RETROSPOT SUGAR JAM BOWL         6   
1022250  578347     22572      ROCKING HORSE GREEN CHRISTMAS          2   
401834   527910     22952     60 CAKE CASES VINTAGE CHRISTMAS        24   
67024    495357     22185          SLATE TILE NATURAL HANGING         6   
937903   572288     22983                 CARD BILLBOARD FONT        12   
1027588  578842     22547               MINI JIGSAW DINOSAUR          1   
726797   554280        C2                            CARRIAGE         1   

                InvoiceDate  Price  Customer ID         Country  
611391  2011-02-09 12:46:00   2.46          NaN  United Kingdom  
664642  2011-03-30 12:00:00   9.95      17